In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import scanpy as sc
import sys

sys.path.append("../../utils/")
import analysis

In [ ]:
%load_ext lab_black

In [ ]:
path_hlca_full = "/ictstr01/groups/ml01/projects/2024_hlca_v2_raphael.kfurirubens/hlca_v1/hlca_v1_full.h5ad"

In [ ]:
adata_full = sc.read_h5ad(path_hlca_full, backed="r")

subset to pericytes/smooth muscle:

In [ ]:
mask = (
    (adata_full.obs.ann_level_2 == "Smooth muscle")
    | (adata_full.obs.ann_level_4 == "Pericytes")
    | (adata_full.obs.transf_ann_level_2_label == "Smooth muscle")
    | (adata_full.obs.transf_ann_level_4_label == "Pericytes")
)

In [ ]:
adata = adata_full[mask, :].to_memory()

set gene names as var row name (instead of gene id):

In [ ]:
adata.var["ensembl_id"] = adata.var.index
adata.var.index = adata.var.feature_name

remove raw (not needed):

In [ ]:
del adata.raw

merge HLCA core manual annotation with HLCA extension transfered labels:

In [ ]:
for lev in range(1, 6):
    adata.obs[f"ann_level_{lev}_merged"] = adata.obs[f"ann_level_{lev}"].tolist()
    adata.obs.loc[
        adata.obs.core_or_extension == "extension", f"ann_level_{lev}_merged"
    ] = adata.obs.loc[
        adata.obs.core_or_extension == "extension", f"transf_ann_level_{lev}_label"
    ]
    adata.obs[f"ann_level_{lev}_clean"] = analysis.clean_up_df_column(
        adata.obs, f"ann_level_{lev}_merged", 500
    )

recalculate kNN graph and umap:

In [ ]:
sc.pp.neighbors(adata, use_rep="X_scanvi_emb", n_neighbors=30)

In [ ]:
sc.tl.umap(adata)

perform clustering:

In [ ]:
sc.tl.leiden(adata, resolution=0.5)

plot annotations and clusters:

In [ ]:
sc.pl.umap(
    adata,
    color=[f"ann_level_{lev}_clean" for lev in [2, 3, 4]]
    + ["leiden", "core_or_extension"],
    frameon=False,
    ncols=2,
    wspace=0.7,
)

In [ ]:
sc.pl.umap(adata, color=["HIGD1B", "LAMC3", "study"])

create obs column that is a combination of ann and study, then create obs column that is a combination of ann, study and lung condition:

In [ ]:
for lev in [2, 4]:
    adata.obs[f"ann_level_{lev}_study"] = [
        f"{ct}_{study}"
        for ct, study in zip(adata.obs[f"ann_level_{lev}_clean"], adata.obs.study)
    ]
    adata.obs[f"ann_level_{lev}_study"] = analysis.clean_up_df_column(
        adata.obs, f"ann_level_{lev}_study", 50
    )
    adata.obs[f"ann_level_{lev}_study_lung_condition"] = [
        f"{ct}_{study}"
        for ct, study in zip(
            adata.obs[f"ann_level_{lev}_study"], adata.obs.lung_condition
        )
    ]
    adata.obs[f"ann_level_{lev}_study_lung_condition"] = analysis.clean_up_df_column(
        adata.obs, f"ann_level_{lev}_study_lung_condition", 10
    )

also create combination of study and cluster:

In [ ]:
adata.obs["leiden_study"] = [
    f"{cl} ({study}" for cl, study in zip(adata.obs.leiden, adata.obs.study)
]
adata.obs["leiden_study"] = analysis.clean_up_df_column(adata.obs, "leiden_study", 0)

and of study and cluster and condition

In [ ]:
adata.obs["leiden_study_condition"] = [
    f"{cl_st}, {cond})"
    for cl_st, cond in zip(adata.obs.leiden_study, adata.obs.lung_condition)
]
adata.obs["leiden_study_condition"] = analysis.clean_up_df_column(
    adata.obs, "leiden_study_condition", 100
)

Check donor composition per cluster:

In [ ]:
colors = [  # zeileis_28
    "#023fa5",
    "#7d87b9",
    "#bec1d4",
    "#d6bcc0",
    "#bb7784",
    "#8e063b",
    "#4a6fe3",
    "#8595e1",
    "#b5bbe3",
    "#e6afb9",
    "#e07b91",
    "#d33f6a",
    "#11c638",
    "#8dd593",
    "#c6dec7",
    "#ead3c6",
    "#f0b98d",
    "#ef9708",
    "#0fcfc0",
    "#9cded6",
    "#d5eae7",
    "#f3e1eb",
    "#f6c4e1",
    "#f79cd4",
    # these last ones were added:
    "#7f7f7f",
    "#c7c7c7",
    "#1CE6FF",
    "#336600",
]

# Create a ListedColormap (discrete colors, no interpolation)
custom_cmap = mcolors.ListedColormap(colors)

In [ ]:
fig, ax = plt.subplots(figsize=(3, 4))
pd.crosstab(adata.obs.leiden, adata.obs.donor_id).plot(
    kind="bar",
    stacked=True,
    ax=ax,
    legend=False,
    cmap=custom_cmap,  # "tab20",
)
ax.spines[["top", "right"]].set_visible(False)
# leg = plt.legend(loc=(1.01, 0), frameon=False, title="Donor:", fontsize=9)
# leg._legend_box.align = "left"
ax.set_ylabel("N cells")
ax.set_xlabel("Cluster")
ax.set_title("Donor composition per cluster")
plt.show()

In [ ]:
for cl in adata.obs.leiden.unique():
    cl_subject_count = (
        adata.obs.loc[adata.obs.leiden == cl, "donor_id"].value_counts().sort_values()
    )
    cl_subject_fract = cl_subject_count / cl_subject_count.sum()
    print(
        f"Cluster {cl}, fraction of cells from most abundant donor: {round(sum(cl_subject_fract[-1:]),2)}"
    )

simplify so that for clusters with normal expression, we show the cluster as a whole

In [ ]:
def simplifier(x):
    # clusters unrelated to disease are 0,1,3,6 (note that cluster 5 is largely made up of a single donor and therefore also quasi-ignored)
    if x[0] in ["0", "1", "3", "5", "6"]:
        return x[0]
    else:
        return x

In [ ]:
adata.obs["leiden_study_condition_simpl"] = [
    simplifier(x) for x in adata.obs.leiden_study_condition
]

look for state observed in Sheppard_2020 IPF

In [ ]:
markers_paper_figure = {
    "Pericyte,\nall states": ["PTN", "HIGD1B"],
    "Pericyte,\nlacking in IPF state": ["LAMC3", "RGN", "PAG1", "TBX3"],
    "SM, also in IPF pericytes": ["FRZB", "MAP3K7CL"],
    "SM, not in pericytes": ["PLN"],
}

In [ ]:
sc.pl.dotplot(
    adata, var_names=markers_paper_figure, groupby="leiden", standard_scale="var"
)

In [ ]:
adata.obs["lung_condition_clean"] = analysis.clean_up_df_column(
    adata.obs, "lung_condition", 200
)

In [ ]:
fig, ax = plt.subplots(figsize=(3, 4))
pd.crosstab(adata.obs.leiden, adata.obs.lung_condition_clean).plot(
    kind="bar", stacked=True, cmap="tab20", ax=ax
)
ax.spines[["top", "right"]].set_visible(False)
leg = plt.legend(loc=(1.01, 0), frameon=False, title="Lung condition:", fontsize=9)
leg._legend_box.align = "left"
ax.set_ylabel("N cells")
ax.set_xlabel("HLCA smooth muscle/pericyte cluster")
plt.show()

In [ ]:
adata.obs["study_clean"] = analysis.clean_up_df_column(adata.obs, "study", 200)

In [ ]:
# Assuming n=3 (adjust this value as needed)
min_n = 10

# Get count of appearances for each donor within each group
test = (adata.obs.groupby(["leiden", "donor_id"])).agg(
    {"donor_id": "count", "smoking_status": "first"}
)
test = test.loc[test.donor_id > min_n, :]
# .groupby(["leiden", "smoking_status"]).agg(
#     donors_with_n_instances=("donor_id", lambda x: sum(x.value_counts() >= n))
# )

In [ ]:
smoking_status_per_cluster = pd.crosstab(
    test.index.get_level_values(0), test.smoking_status
)
smoking_status_per_cluster_normalized = smoking_status_per_cluster.divide(
    smoking_status_per_cluster.sum(axis=1), axis=0
)

In [ ]:
smoking_status_per_cluster_normalized

In [ ]:
smoking_status = [
    x if not pd.isnull(x) else "NA" for x in adata.obs.smoking_status.tolist()
]
fig, ax = plt.subplots(figsize=(3, 4))
pd.crosstab(adata.obs.leiden, smoking_status).plot(
    kind="bar",
    stacked=True,
    ax=ax,  # palette={'NA':'grey','active':'blue','former':'lightblue','never':'orange','hist of marijuana use':'black'}, ax=ax
)
ax.spines[["top", "right"]].set_visible(False)
leg = plt.legend(loc=(1.01, 0), frameon=False, title="Study:", fontsize=9)
leg._legend_box.align = "left"
ax.set_ylabel("N cells")
ax.set_xlabel("Cluster")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(3, 4))
pd.crosstab(adata.obs.leiden, adata.obs.study_clean).plot(
    kind="bar", stacked=True, cmap=custom_cmap, ax=ax
)
ax.spines[["top", "right"]].set_visible(False)
leg = plt.legend(loc=(1.01, 0), frameon=False, title="Study:", fontsize=9)
leg._legend_box.align = "left"
ax.set_ylabel("N cells")
ax.set_xlabel("Cluster")
plt.show()

In [ ]:
sc.pl.dotplot(
    adata,
    var_names=markers_paper_figure,
    groupby="leiden_study_condition_simpl",
    standard_scale="var",
    # dendrogram=True,
)